In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [15]:
from sklearn.datasets import load_breast_cancer
X,y=load_breast_cancer().data,load_breast_cancer().target


In [16]:
X=pd.DataFrame(X,columns=load_breast_cancer().feature_names)
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [17]:
y=pd.DataFrame(y,columns=['target'])
y.head()

,target
0,0
1,0
2,0
3,0
4,0


In [18]:
y.value_counts()

target
1         357
0         212
Name: count, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split


In [21]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


In [22]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [23]:
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [44]:
class dataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32).to(device)
        self.y=torch.tensor(y.values.squeeze(),dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self,index):
        return self.X[index], self.y[index]

In [49]:
training_data=dataset(X_train,y_train)
testing_data=dataset(X_test,y_test)

In [46]:
train_dataloader=DataLoader(training_data,batch_size=8,shuffle=True)
testing_data=DataLoader(testing_data,batch_size=8,shuffle=True)

In [47]:
hidden_neorons=10
class Model(nn.Module):
    def __init__(self,):
        super(Model,self).__init__()

        self.input_layer=nn.Linear(X.shape[1],hidden_neorons)
        self.linear=nn.Linear(hidden_neorons,1)
        self.sigmoid=nn.Sigmoid()
    
    def forward(self,x):
        x=self.input_layer(x)
        x=self.linear(x)
        x=self.sigmoid(x)
        return x
    
model=Model().to(device)

In [48]:
summary(model,(X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 10]             310
            Linear-2                    [-1, 1]              11
           Sigmoid-3                    [-1, 1]               0
Total params: 321
Trainable params: 321
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [50]:
criterion=nn.BCELoss()
optimizer=Adam(model.parameters(),lr=1e-3)

In [54]:
total_loss_train_plot=[]
total_loss_validation_plot=[]
total_acc_train_plot=[]
total_acc_test_plot=[]


epochs=10

for epoch in range(epochs):
    total_acc_train=0
    total_train_loss=0
    total_acc_test=0
    total_test_loss=0
    num_batches_train = 0
    num_batches_test = 0

    for data in train_dataloader:
        inputs, labels=data

        optimizer.zero_grad()
        
        prediction=model(inputs).view(-1)
        labels=labels.view(-1)
        
        batch_loss=criterion(prediction,labels)
        
        batch_loss.backward()
        optimizer.step()

        total_train_loss+=batch_loss.item()

        acc=(prediction.round()==labels).float().mean()
        total_acc_train+=acc.item()
        num_batches_train+=1

    # Validation loop
    model.eval()
    with torch.no_grad():
        for data in testing_data:
            inputs, labels=data
            prediction=model(inputs).view(-1)
            labels=labels.view(-1)
            batch_loss=criterion(prediction,labels)
            total_test_loss+=batch_loss.item()
            acc=(prediction.round()==labels).float().mean()
            total_acc_test+=acc.item()
            num_batches_test+=1
    model.train()
    
    print(f'Epoch {epoch+1}/{epochs}')
    print(f'Training Loss: {total_train_loss/num_batches_train:.4f}, Training Accuracy: {total_acc_train/num_batches_train:.4f}')
    print(f'Validation Loss: {total_test_loss/num_batches_test:.4f}, Validation Accuracy: {total_acc_test/num_batches_test:.4f}\n')
    
    total_loss_train_plot.append(total_train_loss/num_batches_train)
    total_loss_validation_plot.append(total_test_loss/num_batches_test)
    total_acc_train_plot.append(total_acc_train/num_batches_train)
    total_acc_test_plot.append(total_acc_test/num_batches_test)

Epoch 1/10
Training Loss: 0.1215, Training Accuracy: 0.9671
Validation Loss: 0.1266, Validation Accuracy: 0.9561

Epoch 2/10
Training Loss: 0.1015, Training Accuracy: 0.9756
Validation Loss: 0.1118, Validation Accuracy: 0.9561

Epoch 3/10
Training Loss: 0.0899, Training Accuracy: 0.9803
Validation Loss: 0.1020, Validation Accuracy: 0.9561

Epoch 4/10
Training Loss: 0.0826, Training Accuracy: 0.9821
Validation Loss: 0.0947, Validation Accuracy: 0.9649

Epoch 5/10
Training Loss: 0.0775, Training Accuracy: 0.9843
Validation Loss: 0.0903, Validation Accuracy: 0.9649

Epoch 6/10
Training Loss: 0.0719, Training Accuracy: 0.9846
Validation Loss: 0.0876, Validation Accuracy: 0.9737

Epoch 7/10
Training Loss: 0.0688, Training Accuracy: 0.9868
Validation Loss: 0.0845, Validation Accuracy: 0.9737

Epoch 8/10
Training Loss: 0.0664, Training Accuracy: 0.9846
Validation Loss: 0.0821, Validation Accuracy: 0.9737

Epoch 9/10
Training Loss: 0.0647, Training Accuracy: 0.9868
Validation Loss: 0.0809, Val